# XGBoost를 활용한 암종 분류 (EXP-374 파이프라인)

`EXP-374`(Issue #374, 공식 기록 OOF Macro F1 `0.4267909268459148`)의 실제 학습
파이프라인을 그대로 재현하는 Notebook입니다. 초기 baseline
(`[Baseline]_XGB를 활용한 암종 분류 AI 모델 개발.ipynb`)을 저장소 계약에 맞게
정리한 이전 버전(Issue #10)은 단순 mutation-presence 인코딩만 사용했지만, 이
버전은 팀이 현재 채택 중인 최고 검증 pipeline을 코드 중복 없이 그대로
호출합니다.

> 이 Notebook은 새로운 실험이 아니라 **이미 공식 기록된 EXP-374를 코드로
> 재현**합니다. `scripts/run_exp374_stop_isoform_residue_mask.py`와
> `scripts/run_hotspot_xgb.py`, `scripts/run_exp449_lightgbm_exp374.py`의
> feature 구성 함수를 그대로 import해서 사용하므로, 로직이 두 곳에서
> 따로 관리되지 않습니다. `RUN_MODE`는 계속 `"explore"`이며 새 EXP-ID를
> 발급하지 않습니다.

## 0. EXP-374 파이프라인 개요

### 대회 목적

- 바이오 데이터를 기반으로 한 AI 기술의 문제 해결 능력을 탐구하는 것을 목표로 합니다. 이 대회는 바이오 분야에서 AI 활용의 저변을 확대하고, 복잡한 바이오 데이터를 효율적으로 분석 및 해석할 수 있는 AI 알고리즘 개발에 초점을 맞추고 있습니다.
- 본 대회의 구체적인 과제는 암환자 유전체 데이터의 변이 정보를 활용하여 암종을 분류하는 AI 모델을 개발하는 것입니다.

### 이 Notebook이 재현하는 pipeline (EXP-374)

- **파서**: stop-notation-invariant v2.1.0(`*`/`X`/`Ter` 등 표기 차이를 하나의 정지코돈 의미로 정규화, Issue #369)
- **Isoform 마스크**: Ensembl release 116 semantic mask로 residue-position 기여를 신뢰 가능한 isoform 매치에만 한정(Issue #313)
- **Pathway family**: Sanchez-Vega 10-pathway 카탈로그 기반 burden·mutation-type composition을 **fold별로 fold train에만 fit**하는 fold-safe 방식(Issue #229)
- **Hotspot**: `extended_34` hotspot 테이블 + stop 표기 정규화
- **모델**: XGBoost, `n_estimators=500`, `max_depth=6`, `learning_rate=0.05`, balanced sample weight, canonical 5-fold seed 42

이 구성 요소는 전부 저장소의 기존 함수를 그대로 호출하며, 이 Notebook 안에서
새로 구현하지 않습니다 — 로직이 두 곳에서 따로 관리되면서 실제 실험과 달라질
위험을 없애기 위해서입니다.

## 1. 실행 전 준비

저장소 루트에서 `uv sync --frozen`을 한 번 실행하고 VS Code/Jupyter kernel로 `.venv`의 Python을 선택합니다. 원본 CSV는 저장소의 `data/raw/`에 포함되어 있습니다.

전체 5-fold 학습을 실행하므로, 이 Notebook을 처음부터 끝까지 실행하는 데는
수 분에서 십수 분이 걸릴 수 있습니다(하드웨어에 따라 다름).

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from IPython.display import display
from scipy import sparse
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

from open_cancer.checkpoint_selection import (
    audit_xgboost_validation_iterations,
    predict_xgboost_at_iteration,
)
from open_cancer.constants import CLASS_LABELS, PROBABILITY_COLUMNS
from open_cancer.experiment import resolve_experiment_context
from open_cancer.feature_family import drop_named_base_features
from open_cancer.hotspot_features import build_hotspot_augmented_features, resolve_hotspot_config
from open_cancer.isoform_position_mask import resolve_isoform_position_mask_from_config
from open_cancer.isoform_relative_position import resolve_isoform_relative_position_from_config
from open_cancer.mutation_features import (
    resolve_position_features_from_config,
    resolve_position_options_from_config,
)
from open_cancer.robust_mutation_parser import (
    STOP_NOTATION_PARSER_CONTRACT,
    normalize_stop_notation_token,
    parse_stop_notation_invariant_cell,
)
from open_cancer.validation import validate_competition_data, validate_submission

SEED = 42
N_SPLITS = 5
RUN_MODE = "explore"  # 이 Notebook은 EXP-374를 재현만 하므로 "explore"로 고정합니다.
PARENT_EXPERIMENT_ID = "EXP-374"
PARENT_OFFICIAL_OOF_MACRO_F1 = 0.4267909268459148
USE_BALANCED_SAMPLE_WEIGHT = True
N_JOBS = max(1, min(8, os.cpu_count() or 1))

experiment_context = resolve_experiment_context(RUN_MODE, cwd=Path.cwd())

random.seed(SEED)
np.random.seed(SEED)

{
    "python_seed": SEED,
    "xgboost_version": xgb.__version__,
    "run_mode": RUN_MODE,
    "branch": experiment_context.branch,
    "parent_experiment_id": PARENT_EXPERIMENT_ID,
    "n_jobs": N_JOBS,
}

In [ ]:
def find_project_root(start: Path) -> Path:
    """pyproject.toml과 PROJECT_CONTEXT.md가 있는 저장소 루트를 찾습니다."""
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "PROJECT_CONTEXT.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("open_cancer 저장소 안에서 Notebook을 실행하세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
FOLD_PATH = PROJECT_ROOT / "data" / "splits" / "stratified_5fold_seed42.csv"
EXP374_CONFIG_PATH = PROJECT_ROOT / "configs" / "exp374_stop_isoform_residue_mask.yaml"

{
    "project_root": str(PROJECT_ROOT),
    "train": str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
    "test": str(TEST_PATH.relative_to(PROJECT_ROOT)),
    "fold": str(FOLD_PATH.relative_to(PROJECT_ROOT)),
    "exp374_config": str(EXP374_CONFIG_PATH.relative_to(PROJECT_ROOT)),
}

## 2. 원본 데이터 검증과 로드

파일을 읽기 전에 프로젝트의 고정 데이터 계약을 검증합니다. 그다음 모든 변이 값을 문자열로 유지하고, 원본 train 순서를 보존하면서 공용 fold를 `ID`로 연결합니다.

In [ ]:
data_summary = validate_competition_data(
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
)

train_raw = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
test = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype=str,
    keep_default_na=False,
)
folds = pd.read_csv(FOLD_PATH, dtype={"ID": str, "fold": int})

train = train_raw.merge(folds, on="ID", how="left", validate="one_to_one", sort=False)
if not train["ID"].equals(train_raw["ID"]):
    raise ValueError("fold 병합 과정에서 train 행 순서가 바뀌었습니다.")
if train["fold"].isna().any() or set(train["fold"]) != set(range(N_SPLITS)):
    raise ValueError("공용 fold가 모든 train ID에 0~4로 배정되지 않았습니다.")
if not sample_submission["ID"].equals(test["ID"]):
    raise ValueError("sample_submission과 test의 ID 값 또는 순서가 다릅니다.")

gene_columns = [column for column in test.columns if column != "ID"]
train_gene_columns = [
    column for column in train.columns if column not in {"ID", "SUBCLASS", "fold"}
]
if gene_columns != train_gene_columns:
    raise ValueError("train/test 유전자 컬럼 이름 또는 순서가 다릅니다.")

data_summary

In [ ]:
data_overview = {
    "train_shape_with_fold": train.shape,
    "test_shape": test.shape,
    "gene_columns": len(gene_columns),
    "fold_counts": train["fold"].value_counts().sort_index().to_dict(),
    "class_counts": train["SUBCLASS"].value_counts().sort_index().to_dict(),
    "train_blank_cells": int((train[gene_columns] == "").to_numpy().sum()),
    "test_blank_cells": int((test[gene_columns] == "").to_numpy().sum()),
}
data_overview

## 3. 타깃 인코딩

`LabelEncoder`를 사용하되, 학습 데이터에서 우연히 얻은 순서에 의존하지 않도록 프로젝트의 고정 26개 클래스에 fit합니다. 예측 확률의 열 순서는 항상 `CLASS_LABELS`와 동일합니다.

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(list(CLASS_LABELS))
if list(label_encoder.classes_) != list(CLASS_LABELS):
    raise ValueError("LabelEncoder 클래스 순서가 프로젝트 고정 순서와 다릅니다.")

y = label_encoder.transform(train["SUBCLASS"]).astype(np.int32)
label_mapping = pd.DataFrame(
    {
        "SUBCLASS": label_encoder.classes_,
        "encoded": np.arange(len(label_encoder.classes_)),
    }
)
display(label_mapping)

## 4. EXP-374 feature 파이프라인

원본 baseline의 단순 mutation-presence 인코딩 대신, EXP-374가 실제로 사용한
전체 feature 구성을 그대로 호출합니다.

- `build_base_features`(아래 셀): `open_cancer.hotspot_features.build_hotspot_augmented_features`를
  EXP-374의 config로 호출해 stop-notation-invariant 파서·hotspot-34·Ensembl
  isoform mask·robust burden aggregate가 반영된 base feature 행렬(`x_all`,
  `x_test`)을 만듭니다. 이 함수는 EXP-374의 config
  (`configs/exp374_stop_isoform_residue_mask.yaml`)를 그대로 읽어 옵션을
  결정하므로, 이 Notebook에서 파서·hotspot 옵션을 다시 입력하지 않습니다 —
  `scripts/run_exp449_lightgbm_exp374.py`에서 이미 검증된 것과 동일한
  호출입니다(현재 이 저장소 브랜치에는 아직 병합되지 않아 직접 import할 수
  없으므로 동일 로직을 그대로 옮겨왔습니다).
- `run_exp374_stop_isoform_residue_mask.build_fold_features`: pathway
  burden·mutation-type composition family를 **fold별로 fold train에만
  fit**하는 fold-safe builder를 반환합니다(Issue #229의 `PathwayMutationTypeFoldBuilder`).
  이 함수는 EXP-374 자체의 스크립트에 있으므로 그대로 import합니다.

In [ ]:
from run_exp374_stop_isoform_residue_mask import build_fold_features


def build_base_features(exp374_config: dict, feature_dir: Path) -> dict:
    """scripts/run_exp449_lightgbm_exp374.py의 base feature 빌더와 동일합니다."""
    hotspot_config = exp374_config.get("hotspots", {})
    hotspots, _evidence, _min_rows = resolve_hotspot_config(hotspot_config)
    selected_position_features = resolve_position_features_from_config(exp374_config)
    position_options = resolve_position_options_from_config(exp374_config)
    position_token_filter, mask_contract = resolve_isoform_position_mask_from_config(
        exp374_config, root=PROJECT_ROOT
    )
    position_token_transformer, relative_contract = resolve_isoform_relative_position_from_config(
        exp374_config, root=PROJECT_ROOT
    )
    position_semantic_contract = relative_contract or mask_contract
    selected_robust_aggregates = tuple(
        exp374_config.get("features", {}).get("robust_aggregates", [])
    )
    return build_hotspot_augmented_features(
        TRAIN_PATH,
        TEST_PATH,
        feature_dir,
        hotspots=hotspots,
        base_feature_options={
            "selected_robust_aggregates": selected_robust_aggregates,
            "selected_position_features": selected_position_features,
            "position_token_filter": position_token_filter,
            "position_token_transformer": position_token_transformer,
            "position_semantic_contract": position_semantic_contract,
            "mutation_cell_parser": parse_stop_notation_invariant_cell,
            "mutation_parser_contract": STOP_NOTATION_PARSER_CONTRACT,
            **position_options,
        },
        hotspot_token_normalizer=normalize_stop_notation_token,
    )


exp374_config = yaml.safe_load(EXP374_CONFIG_PATH.read_text(encoding="utf-8"))
feature_dir = PROJECT_ROOT / "data" / "processed" / "notebook_exp374_pipeline_features"

feature_report = build_base_features(exp374_config, feature_dir)
x_all = sparse.load_npz(feature_dir / "train_features.npz")
x_test = sparse.load_npz(feature_dir / "test_features.npz")
all_feature_names = tuple(
    json.loads((feature_dir / "feature_names.json").read_text(encoding="utf-8"))
)

feature_summary = {
    "base_feature_count": len(all_feature_names),
    "train_matrix_shape": x_all.shape,
    "test_matrix_shape": x_test.shape,
    "train_nonzero": int(x_all.nnz),
    "test_nonzero": int(x_test.nnz),
}
feature_summary

## 5. XGBoost 모델과 공용 5-fold 학습

`configs/exp374_stop_isoform_residue_mask.yaml`의 `model` 섹션에 고정된
하이퍼파라미터를 그대로 사용합니다(EXP-285에서 튜닝된 파라미터는 EXP-374에
의도적으로 포함하지 않습니다). 각 fold마다 pathway family를 fold train에만
fit한 뒤 base feature와 이어붙여 학습합니다(fold-safe).

학습·early stopping 자체는 multi-class log loss 기준으로 진행하되(미분
가능한 objective가 아닌 Macro F1로는 직접 early stopping할 수 없음),
**최종 checkpoint 선택은 config의 `training.checkpoint_selection:
macro_f1_validation` 정책에 따라 validation fold의 Macro F1을 최대화하는
boosting iteration을 별도로 감사해 선택**합니다
(`open_cancer.checkpoint_selection.audit_xgboost_validation_iterations`,
test/Public은 이 감사에 전혀 사용되지 않습니다) — log loss 기준
early-stopping iteration을 그대로 쓰면 EXP-374의 공식 OOF Macro F1과
일치하지 않습니다.

In [ ]:
XGB_PARAMS = dict(exp374_config["model"])
XGB_PARAMS.pop("checkpoint_selection", None)
XGB_PARAMS["num_class"] = len(CLASS_LABELS)
XGB_PARAMS

In [ ]:
fold_builder = build_fold_features()

fold_results: list[dict[str, int | float | None]] = []
oof_proba = np.full((len(train), len(CLASS_LABELS)), np.nan, dtype=np.float64)
test_proba = np.zeros((len(test), len(CLASS_LABELS)), dtype=np.float64)

for fold in range(N_SPLITS):
    valid_mask = train["fold"].eq(fold).to_numpy()
    train_indices = np.flatnonzero(~valid_mask)
    valid_indices = np.flatnonzero(valid_mask)
    y_train_fold, y_valid_fold = y[train_indices], y[valid_indices]
    x_train_base = x_all[train_indices]
    x_valid_base = x_all[valid_indices]

    extra = fold_builder(
        fold=fold,
        train_indices=train_indices,
        valid_indices=valid_indices,
        base_train=x_train_base,
        base_validation=x_valid_base,
        base_test=x_test,
        base_feature_names=all_feature_names,
        target=y_train_fold,
    )
    x_train_dropped, x_valid_dropped, x_test_dropped, _kept_names = drop_named_base_features(
        x_train_base, x_valid_base, x_test, all_feature_names, extra.base_feature_names_to_drop
    )
    x_train_fold = sparse.hstack([x_train_dropped, extra.train], format="csr", dtype=np.float32)
    x_valid_fold = sparse.hstack([x_valid_dropped, extra.validation], format="csr", dtype=np.float32)
    x_test_fold = sparse.hstack([x_test_dropped, extra.test], format="csr", dtype=np.float32)

    sample_weight = (
        compute_sample_weight(class_weight="balanced", y=y_train_fold)
        if USE_BALANCED_SAMPLE_WEIGHT
        else None
    )

    model = xgb.XGBClassifier(**XGB_PARAMS, random_state=SEED + fold)
    model.fit(
        x_train_fold,
        y_train_fold,
        sample_weight=sample_weight,
        eval_set=[(x_valid_fold, y_valid_fold)],
        verbose=False,
    )
    if not np.array_equal(model.classes_, np.arange(len(CLASS_LABELS))):
        raise ValueError(f"fold {fold} 모델의 확률 클래스 순서가 다릅니다.")

    checkpoint_audit = audit_xgboost_validation_iterations(
        model,
        x_valid_fold,
        y_valid_fold,
        selection_policy="macro_f1_validation",
    )
    selected_iteration = int(checkpoint_audit["selected_checkpoint"]["iteration"])
    valid_proba = predict_xgboost_at_iteration(model, x_valid_fold, selected_iteration).astype(np.float64)
    fold_test_proba = predict_xgboost_at_iteration(model, x_test_fold, selected_iteration).astype(np.float64)
    oof_proba[valid_indices] = valid_proba
    test_proba += fold_test_proba / N_SPLITS

    valid_predictions = valid_proba.argmax(axis=1)
    fold_macro_f1 = f1_score(y_valid_fold, valid_predictions, average="macro")
    fold_results.append(
        {
            "fold": fold,
            "macro_f1": float(fold_macro_f1),
            "best_iteration": selected_iteration,
            "train_rows": len(train_indices),
            "valid_rows": len(valid_indices),
        }
    )
    print(f"fold={fold} macro_f1={fold_macro_f1:.6f} selected_iteration={selected_iteration}")

if np.isnan(oof_proba).any():
    raise ValueError("OOF 확률에 채워지지 않은 행이 있습니다.")

## 6. OOF Macro F1 평가와 EXP-374 공식 기록 대조

5개 validation fold를 모두 합친 OOF 예측으로 공식 지표와 같은 Macro F1을
계산하고, EXP-374의 공식 기록(`0.4267909268459148`)과 직접 비교합니다.

이 Notebook은 저장된 checkpoint를 재추론하는 것이 아니라 **처음부터 다시
학습**하므로, `INFERENCE_VERIFIED`급 byte-identical 일치는 기대하지 않습니다.
이 저장소는 XGBoost `tree_method=hist`가 플랫폼(OS·CPU 아키텍처)에 따라
처음부터 재학습할 때 완전히 결정론적이지 않다는 것을 이미 별도로 감사·기록해
두었습니다
([`xgboost_hist_cross_platform_reproducibility.md`](../reports/analysis/xgboost_hist_cross_platform_reproducibility.md),
EXP-219 macOS→Windows 재학습에서 OOF Macro F1 차이 `-0.0012566`). EXP-374의
공식 기록은 macOS arm64에서 만들어졌고, 이 Notebook은 그와 다른 플랫폼에서
실행될 수 있으므로, 그 문서가 이미 관측한 것과 같은 크기(수천분의 1
수준)의 차이는 파이프라인 버그가 아니라 예상된 cross-platform 재학습
변동으로 간주합니다. 그보다 훨씬 큰 차이가 나면 파이프라인 재현 자체에
문제가 있다는 뜻이므로 아래 셀이 실패합니다.

In [ ]:
oof_predictions = oof_proba.argmax(axis=1)
oof_macro_f1 = f1_score(y, oof_predictions, average="macro")
class_report = classification_report(
    y,
    oof_predictions,
    labels=np.arange(len(CLASS_LABELS)),
    target_names=CLASS_LABELS,
    output_dict=True,
    zero_division=0,
)
class_f1 = pd.DataFrame(
    {
        "SUBCLASS": CLASS_LABELS,
        "f1": [class_report[label]["f1-score"] for label in CLASS_LABELS],
        "support": [int(class_report[label]["support"]) for label in CLASS_LABELS],
    }
)
fold_metrics = pd.DataFrame(fold_results)
macro_f1_delta = oof_macro_f1 - PARENT_OFFICIAL_OOF_MACRO_F1
# EXP-219의 관측된 macOS->Windows 재학습 차이(-0.0012566)를 참고한 완화된 허용치.
# reports/analysis/xgboost_hist_cross_platform_reproducibility.md 참고.
CROSS_PLATFORM_MACRO_F1_TOLERANCE = 0.005
print(f"이 Notebook의 OOF Macro F1: {oof_macro_f1:.10f}")
print(f"EXP-374 공식 기록:          {PARENT_OFFICIAL_OOF_MACRO_F1:.10f}")
print(f"차이:                       {macro_f1_delta:+.10f}")
if abs(macro_f1_delta) > CROSS_PLATFORM_MACRO_F1_TOLERANCE:
    raise ValueError(
        "이 Notebook의 OOF Macro F1이 EXP-374 공식 기록과 cross-platform 허용치"
        f"(+-{CROSS_PLATFORM_MACRO_F1_TOLERANCE})를 넘어 차이 납니다 -- "
        "파이프라인 재현 자체에 차이가 있을 가능성이 높습니다."
    )
elif abs(macro_f1_delta) > 1e-6:
    print(
        "-> EXP-219 감사에서 관측한 것과 같은 크기의 cross-platform XGBoost hist "
        "재학습 변동으로 판단됩니다(파이프라인 버그 아님)."
    )
display(fold_metrics)
display(class_f1)

## 7. 테스트 추론과 제출 파일 생성

각 fold 모델의 test 확률을 동일 가중치로 평균하고, 가장 높은 확률의 클래스를 원래 `SUBCLASS` 문자열로 복원합니다. 제출 저장 직후 프로젝트 검증 함수를 실행합니다.

In [ ]:
artifact_slug = "notebook_exp374_pipeline_reproduction"
oof_dir = PROJECT_ROOT / "oof"
preds_dir = PROJECT_ROOT / "preds"
submissions_dir = PROJECT_ROOT / "submissions"
report_dir = PROJECT_ROOT / "reports" / artifact_slug
for directory in (oof_dir, preds_dir, submissions_dir, report_dir):
    directory.mkdir(parents=True, exist_ok=True)

oof_frame = pd.DataFrame(
    {
        "ID": train["ID"],
        "SUBCLASS_TRUE": train["SUBCLASS"],
        "SUBCLASS_PRED": label_encoder.inverse_transform(oof_predictions),
        "FOLD": train["fold"].astype(int),
    }
)
oof_frame.loc[:, list(PROBABILITY_COLUMNS)] = oof_proba

test_probability_frame = pd.DataFrame({"ID": test["ID"]})
test_probability_frame.loc[:, list(PROBABILITY_COLUMNS)] = test_proba

submission = sample_submission.copy()
submission["SUBCLASS"] = label_encoder.inverse_transform(test_proba.argmax(axis=1))

oof_path = oof_dir / f"{artifact_slug}.csv"
test_probability_path = preds_dir / f"{artifact_slug}_test_proba.csv"
submission_path = submissions_dir / f"{artifact_slug}.csv"
oof_frame.to_csv(oof_path, index=False, lineterminator="\n")
test_probability_frame.to_csv(test_probability_path, index=False, lineterminator="\n")
submission.to_csv(submission_path, index=False, lineterminator="\n")

submission_validation = validate_submission(submission_path, TEST_PATH)
notebook_run_summary = {
    "parent_experiment_id": PARENT_EXPERIMENT_ID,
    "seed": SEED,
    "balanced_sample_weight": USE_BALANCED_SAMPLE_WEIGHT,
    "xgboost_version": xgb.__version__,
    "xgb_params": XGB_PARAMS,
    "fold_results": fold_results,
    "oof_macro_f1": float(oof_macro_f1),
    "oof_macro_f1_delta_vs_exp374": float(macro_f1_delta),
    "oof_macro_f1_within_cross_platform_tolerance": abs(macro_f1_delta)
    <= CROSS_PLATFORM_MACRO_F1_TOLERANCE,
    "data_files": data_summary["files"],
    "feature_order_sha256": data_summary["feature_order_sha256"],
    "submission_validation": submission_validation,
}
summary_path = report_dir / "notebook_run.json"
summary_path.write_text(
    json.dumps(notebook_run_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"OOF: {oof_path.relative_to(PROJECT_ROOT)}")
print(f"Test probabilities: {test_probability_path.relative_to(PROJECT_ROOT)}")
print(f"Submission: {submission_path.relative_to(PROJECT_ROOT)}")
print(f"Summary: {summary_path.relative_to(PROJECT_ROOT)}")
display(submission.head())
submission_validation

## 8. 이 Notebook의 위치

이 Notebook은 팀이 이미 공식 기록한 `EXP-374`를 코드로 재현한 결과물입니다 —
새 아이디어를 시험하는 용도가 아닙니다.

- 실제 리더보드 제출·재현성 증빙(`INFERENCE_VERIFIED`/`TRAINING_VERIFIED`)의
  근거는 `configs/exp374_stop_isoform_residue_mask.yaml`,
  `scripts/run_exp374_stop_isoform_residue_mask.py`,
  `reproducibility/exp374_stop_isoform_residue_mask/`이며, 이 Notebook의
  실행 결과가 아닙니다.
- 새로운 feature나 모델 아이디어를 시험할 때는 이 Notebook을 복제해
  프로토타입한 뒤, 채택되면 새 Experiment Issue를 만들고 로직을
  `configs/`·`scripts/run_expNNN_<slug>.py`로 옮겨 공식 실험으로
  실행합니다(`notebooks/README.md` 참고).
- `PROJECT_CONTEXT.md` 기준 "최종 제출 후보"는 `TRAINING_VERIFIED`(비작성자의
  독립 재학습 검증)가 필요합니다. 이 Notebook의 재현 실행은 작성자 본인의
  재실행이므로 `TRAINING_VERIFIED` 승격 증빙으로 사용할 수 없습니다.